# Rewards for healthier recommendation algorithms
COMSCI/ECON 206 · Computational Microeconomics · Autumn 2026 Session 1

**Question:** Can a regulator's reward change competing apps' recommendation choices?

Run all cells; Python standard library only. All payoffs are hypothetical, with no human data, fitted parameters or measured health effects.

## Model and prediction
Two apps simultaneously choose C (assumed healthy) or D (assumed addictive), with complete information. Each maximizes its own payoff; the regulator sets a nonnegative reward r before play. A pure strategy is an action; a mixed strategy is a probability of C. The regulator is exogenous here.

| A / B | C | D |
|---|---|---|
| C | (3+r−c, 3+r−c) | (1+r−c, 4) |
| D | (4, 1+r−c) | (2, 2) |

Baseline c=0 reproduces the [existing game](https://huggingface.co/spaces/dku-comsci-econ206-2026/Recommendation_Algorithms) checked on September 13, 2026. The modification adds cost c=0.5 to each C player's payoff; the game does not include this cost.

Against either opponent action, switching from D to C changes payoff by r−c−1. Thus DD is the unique Nash equilibrium below r=1+c, and CC is unique above it. At equality all four pure profiles, and every mixed-strategy profile, are equilibria. The code enumerates pure equilibria through unilateral deviations, independently checking this prediction. Comparisons use tolerance 1e-9.

In [1]:
from itertools import product
from math import isfinite
import platform

print("Python", platform.python_version(), "| standard library only | deterministic; no seed")

def game(r, c=0):
    if any(not isfinite(x) or x < 0 for x in (r, c)):
        raise ValueError("Reward and cost must be finite and nonnegative.")
    return {"CC": (3+r-c, 3+r-c), "CD": (1+r-c, 4),
            "DC": (4, 1+r-c), "DD": (2, 2)}

def equilibria(r, c=0):
    pay = game(r, c)
    return [a+b for a, b in product("CD", repeat=2)
            if pay[a+b][0] >= pay[("D" if a == "C" else "C")+b][0] - 1e-9
            and pay[a+b][1] >= pay[a+("D" if b == "C" else "C")][1] - 1e-9]

Python 3.13.15 | standard library only | deterministic; no seed


## Baseline and one modification
Metric: the set of pure Nash equilibria at each reward. The regulator pays r per C player; transfers are not a measure of net social welfare.

In [2]:
print("cost  reward  pure equilibria")
for c in (0, 0.5):
    for r in (0, 1, 1.5, 2):
        print(f"{c:4.1f}  {r:6.1f}  {', '.join(equilibria(r, c))}")
print("Baseline payoffs at r=2:", game(2))

cost  reward  pure equilibria
 0.0     0.0  DD
 0.0     1.0  CC, CD, DC, DD
 0.0     1.5  CC
 0.0     2.0  CC
 0.5     0.0  DD
 0.5     1.0  DD
 0.5     1.5  CC, CD, DC, DD
 0.5     2.0  CC
Baseline payoffs at r=2: {'CC': (5, 5), 'CD': (3, 4), 'DC': (4, 3), 'DD': (2, 2)}


## Verification
Check the corrected baseline ordering, both sides of each threshold, exact ties, the game's upper reward bound, and invalid inputs. The equilibrium routine also accepts finite rewards above the game's UI limit of 5.

In [3]:
assert game(0) == {"CC": (3, 3), "CD": (1, 4), "DC": (4, 1), "DD": (2, 2)}
for c in (0, 0.5):
    assert equilibria(0, c) == ["DD"]
    assert equilibria(1+c-0.001, c) == ["DD"]
    assert equilibria(1+c, c) == ["CC", "CD", "DC", "DD"]
    assert equilibria(1+c+0.001, c) == ["CC"]
    assert equilibria(5, c) == ["CC"]
for r, c in [(-1, 0), (0, -1), (float("nan"), 0), (0, float("inf"))]:
    try:
        game(r, c)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid input was accepted")
print("All checks passed.")

All checks passed.


## Results and research limits
The executed baseline yields DD at r=0, all four pure equilibria at r=1, and CC at r=1.5 and 2. Adding cost 0.5 moves the threshold to 1.5: r=1 now yields DD, r=1.5 gives ties, and r=2 yields CC. At r=2 in the baseline, each app receives 5 and the regulator spends 4 in total. These are model calculations, not observed platform responses.

**Q2 (planned):** Measure user-reported benefit and regret separately from attention time. Pilot the measures before setting an eligibility rule for rewards; ratings alone do not validate mental-health benefits. This notebook assumes the C/D labels and implements the incentive calculation only, as does the game.

**Q3 (planned):** Randomly disclose policy labels versus withhold them, keeping the available apps/content comparable. Measure actual choice and later regret; self-reported reasons alone cannot establish the mechanism. A null choice effect would challenge an information-only explanation but would not identify temptation uniquely. No behavioral experiment has been run here.

The model assumes reliable labels, fixed payoffs, identical apps and no budget constraint. False labels, heterogeneous users or strategic manipulation could change its predictions. The cost experiment tests one assumption only.

## Source and AI assistance
Nash, J. F. (1950). Equilibrium points in n-person games. *PNAS*, 36(1), 48–49. https://doi.org/10.1073/pnas.36.1.48

Payoff source: the author's linked Hugging Face game, inspected September 13, 2026. The cost extension is hypothetical. Framing follows the supplied research proposal.

Codex assisted with code and explanatory text on September 13, 2026, following the supplied draft. Request: generate concise GitHub and Google Colab files from the draft and assignment requirements. Assistance included reproducing the game's model, adding the cost experiment, and checking outputs. Saved outputs were generated by running these code cells sequentially in a fresh local Python process. The author must review the files and record their own verification and accepted/revised suggestions. This does not document independent initial reasoning or peer review.